# TimeSformer MAE Training - IMPROVED ASYMMETRIC MASKING

**NEW MASKING STRATEGY:**
- Mask frames 0-1 (BEFORE) and 5-6 (AFTER) **heavily (90%)**
- Keep middle frames 2-4 (CONTEXT) **more visible (50%)**

**WHY THIS WORKS:**
- Forces encoder to learn typology-distinguishing features in before/after states
- Model must predict important frames from context → learns transformations
- Different typologies have different before→after patterns → better clustering!

## Setup

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import time
from datetime import datetime

from config import get_mae_config, get_small_config
from timesformer_mae import TimeSformerMAE
from dataset import create_dataloaders

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available, using CPU")

## Configuration

In [ ]:
# ============================================================================
# DATA & MODEL SETTINGS
# ============================================================================
DATA_ROOT = r"C:\Users\shrua\OneDrive\Desktop\threshold project\threshold\data"

MODEL_SIZE = 'default'  # 'default' or 'small'

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
NUM_EPOCHS = 100  # Use 100+ for full training, 10 for testing
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
TRAIN_SPLIT = 0.85

# ============================================================================
# SYSTEM SETTINGS
# ============================================================================
NUM_WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_DIR = r"C:\Users\shrua\OneDrive\Desktop\threshold project\threshold\models\output_improved"
SAVE_FREQ = 5  # Save checkpoint every N epochs

# ============================================================================
# PRINT CONFIGURATION
# ============================================================================
print("="*80)
print("IMPROVED MAE TRAINING CONFIGURATION")
print("="*80)
print(f"Model size:     {MODEL_SIZE}")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Batch size:     {BATCH_SIZE}")
print(f"Learning rate:  {LEARNING_RATE}")
print(f"Device:         {DEVICE}")
print(f"Output dir:     {OUTPUT_DIR}")
print("="*80)

In [ ]:
# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'visualizations'), exist_ok=True)
print(f"✓ Output directories created")

## Load Data

In [ ]:
print("Loading dataset...")

train_loader, val_loader = create_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
)

print(f"✓ Data loaded")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Train samples: {len(train_loader.dataset)}")
print(f"  Val samples: {len(val_loader.dataset)}")

## Create Model

In [ ]:
# Get config
if MODEL_SIZE == 'small':
    config = get_small_config()
    print("Using SMALL model (for testing)")
else:
    config = get_mae_config()
    print("Using DEFAULT model (for training)")

# Create model
model = TimeSformerMAE(config)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Model created")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Encoder parameters: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"  Decoder parameters: {sum(p.numel() for p in model.decoder.parameters()):,}")

## Visualize Masking Strategy

In [ ]:
# Get a sample batch
sample_batch = next(iter(train_loader))
sample_imgs = sample_batch['pixel_values'][:1].to(DEVICE)  # Just 1 sample

# Apply masking
model.eval()
with torch.no_grad():
    x = model.encoder.patch_embed(sample_imgs)
    x_masked, mask, ids_restore = model.random_masking(x)

# Visualize mask pattern
mask_np = mask[0].cpu().numpy()  # (896,)
T = 7
P = 128
mask_2d = mask_np.reshape(T, P)  # (7, 128)

# Count visible patches per frame
visible_per_frame = (1 - mask_2d).sum(axis=1)
masked_per_frame = mask_2d.sum(axis=1)

print("\n" + "="*80)
print("MASKING PATTERN ANALYSIS")
print("="*80)
print(f"\nFrame | Visible | Masked | Mask Ratio")
print("-" * 45)
for t in range(T):
    vis = int(visible_per_frame[t])
    mas = int(masked_per_frame[t])
    ratio = mas / P * 100
    frame_type = "BEFORE" if t <= 1 else "AFTER" if t >= 5 else "CONTEXT"
    print(f"  {t}   |   {vis:3d}   |  {mas:3d}   | {ratio:5.1f}% [{frame_type}]")

print(f"\nTotal visible: {visible_per_frame.sum():.0f} / 896 ({visible_per_frame.sum()/896*100:.1f}%)")
print(f"Total masked:  {masked_per_frame.sum():.0f} / 896 ({masked_per_frame.sum()/896*100:.1f}%)")
print("="*80)

# Plot visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Left: Mask pattern heatmap
im = axes[0].imshow(mask_2d, cmap='RdYlGn_r', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Patch Index (0-127)', fontsize=11)
axes[0].set_ylabel('Frame', fontsize=11)
axes[0].set_yticks(range(7))
axes[0].set_yticklabels([f'F{i}' for i in range(7)])
axes[0].set_title('Masking Pattern (Red=Masked, Green=Visible)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0], label='Masked (1) / Visible (0)')

# Add frame type annotations
axes[0].text(135, 0, 'BEFORE', fontsize=10, fontweight='bold', color='red')
axes[0].text(135, 1, 'BEFORE', fontsize=10, fontweight='bold', color='red')
axes[0].text(135, 3, 'CONTEXT', fontsize=10, fontweight='bold', color='blue')
axes[0].text(135, 5, 'AFTER', fontsize=10, fontweight='bold', color='red')
axes[0].text(135, 6, 'AFTER', fontsize=10, fontweight='bold', color='red')

# Right: Bar chart
x_pos = np.arange(T)
axes[1].bar(x_pos, visible_per_frame, color='green', alpha=0.7, label='Visible')
axes[1].bar(x_pos, masked_per_frame, bottom=visible_per_frame, color='red', alpha=0.7, label='Masked')
axes[1].set_xlabel('Frame', fontsize=11)
axes[1].set_ylabel('Number of Patches', fontsize=11)
axes[1].set_title('Patches per Frame', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'F{i}' for i in range(7)])
axes[1].legend()
axes[1].axhline(y=128, color='gray', linestyle='--', alpha=0.5, label='Total=128')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'visualizations', 'masking_strategy.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Masking visualization saved")

## Setup Training

In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=LEARNING_RATE * 0.01,
)

print("✓ Optimizer and scheduler created")

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rate': [],
    'epoch_time': [],
}

best_val_loss = float('inf')

## Training Functions

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device, epoch):
    """
    Train for one epoch.
    """
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")
    
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        
        # Forward pass
        loss, pred, mask = model(pixel_values)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss
        total_loss += loss.item()
        
        # Update progress bar
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss


def validate(model, dataloader, device, epoch):
    """
    Validate the model.
    """
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")
        
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(device)
            
            loss, pred, mask = model(pixel_values)
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss


def save_checkpoint(model, optimizer, epoch, loss, filename):
    """
    Save model checkpoint.
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    torch.save(checkpoint, filename)
    print(f"  Saved: {os.path.basename(filename)}")

## TRAINING LOOP

In [ ]:
# ============================================================================
# RESUME FROM CHECKPOINT (OPTIONAL)
# ============================================================================
RESUME_TRAINING = False
CHECKPOINT_TO_RESUME = r"C:\Users\shrua\OneDrive\Desktop\threshold project\threshold\models\output_improved\checkpoints\checkpoint_epoch_0010.pt"

if RESUME_TRAINING and os.path.exists(CHECKPOINT_TO_RESUME):
    print("="*80)
    print("RESUMING FROM CHECKPOINT")
    print("="*80)
    
    checkpoint = torch.load(CHECKPOINT_TO_RESUME, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    previous_loss = checkpoint.get('loss', 'unknown')
    
    print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"  Previous loss: {previous_loss}")
    print(f"  Resuming from epoch {start_epoch}")
    print(f"  Will train for {NUM_EPOCHS - start_epoch} more epochs")
    print("="*80 + "\n")
else:
    start_epoch = 0
    print("Starting training from scratch (epoch 0)\n")

In [ ]:
print("="*80)
print(f"STARTING TRAINING: {NUM_EPOCHS - start_epoch} epochs")
print("="*80)
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
print()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()
    
    # Train
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch)
    
    # Validate
    val_loss = validate(model, val_loader, DEVICE, epoch)
    
    # Update scheduler
    scheduler.step()
    
    # Record history
    epoch_time = time.time() - epoch_start
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    history['epoch_time'].append(epoch_time)
    
    # Print epoch summary
    print(f"\nEpoch {epoch}/{NUM_EPOCHS-1}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  LR:         {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Time:       {epoch_time:.1f}s")
    
    # Save checkpoint
    if (epoch + 1) % SAVE_FREQ == 0:
        checkpoint_path = os.path.join(
            OUTPUT_DIR, 'checkpoints', f'checkpoint_epoch_{epoch:04d}.pt'
        )
        save_checkpoint(model, optimizer, epoch, val_loss, checkpoint_path)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'best_model.pt')
        save_checkpoint(model, optimizer, epoch, val_loss, best_path)
        print(f"  ⭐ New best model! (val_loss: {val_loss:.4f})")
    
    print("-" * 80)

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total time: {sum(history['epoch_time'])/60:.1f} minutes")

## Save Final Model

In [ ]:
# Save final model
final_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'final_model.pt')
save_checkpoint(model, optimizer, NUM_EPOCHS - 1, history['val_loss'][-1], final_path)

# Save encoder only (for classification fine-tuning)
encoder_path = os.path.join(OUTPUT_DIR, 'timesformer_encoder_pretrained.pt')
torch.save({
    'config': config,
    'encoder_state_dict': model.encoder.state_dict(),
}, encoder_path)

print(f"\n✓ Pretrained encoder saved: {encoder_path}")
print("\n⭐ Use this encoder for classification fine-tuning (next step)")

## Visualize Training Results

In [ ]:
print("Loading BEST model for visualization...")

# Path to best checkpoint
BEST_CHECKPOINT = os.path.join(OUTPUT_DIR, 'checkpoints', 'best_model.pt')

# Load checkpoint
checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")
print(f"  Loss: {checkpoint['loss']:.4f}")
print()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Loss curves
epochs_range = range(len(history['train_loss']))
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Learning rate
axes[1].plot(epochs_range, history['learning_rate'], 'g-', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

# Plot 3: Time per epoch
axes[2].bar(epochs_range, history['epoch_time'], color='orange', alpha=0.7)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Time (seconds)', fontsize=12)
axes[2].set_title('Time per Epoch', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Training curves saved to: {os.path.join(OUTPUT_DIR, 'training_curves.png')}")

## T-SNE Clustering Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from tqdm.notebook import tqdm

print("="*80)
print("EXTRACTING EMBEDDINGS FROM VALIDATION SET")
print("="*80)

# Extract embeddings from validation set
all_embeddings = []
all_labels = []

model.eval()
with torch.no_grad():
    for i, batch in enumerate(tqdm(val_loader, desc="Extracting embeddings")):
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels = batch['labels'].numpy()
        
        # Get embeddings
        x = model.encoder.patch_embed(pixel_values)
        latent = model.encoder(x)
        latent_pooled = latent.mean(dim=1)  # Pool across patches
        
        all_embeddings.append(latent_pooled.cpu().numpy())
        all_labels.append(labels)

embeddings = np.concatenate(all_embeddings)
labels = np.concatenate(all_labels)

print(f"✓ Extracted {len(embeddings)} embeddings")
print(f"  Embedding dimension: {embeddings.shape[1]}")
print()

# t-SNE projection
print("Computing t-SNE projection...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings)
print("✓ t-SNE complete")
print()

# K-means clustering
print("Computing k-means clustering...")
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embeddings)
print("✓ K-means complete")
print()

# Calculate metrics
silhouette = silhouette_score(embeddings_2d, labels)
ari = adjusted_rand_score(labels, clusters)

print(f"Clustering Metrics:")
print(f"  Silhouette Score: {silhouette:.3f} (higher is better)")
print(f"  Adjusted Rand Index: {ari:.3f} (1.0 = perfect match with ground truth)")
print()

# Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', 
          '#ff7f00', '#ffff33', '#a65628', '#f781bf']

# Left: K-means clusters (unsupervised)
for i in range(8):
    mask = clusters == i
    axes[0].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   c=colors[i], label=f'Cluster {i}', alpha=0.6, s=50)
axes[0].set_title(f'K-Means Clustering (Unsupervised)\nARI={ari:.3f}', 
                  fontsize=14, fontweight='bold')
axes[0].legend(ncol=2)
axes[0].grid(alpha=0.3)

# Right: Ground truth typologies
for i in range(8):
    mask = labels == i
    if mask.sum() > 0:
        axes[1].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                       c=colors[i], label=f't{i+1}', alpha=0.6, s=50)
axes[1].set_title(f'Ground Truth Typologies\nSilhouette={silhouette:.3f}', 
                  fontsize=14, fontweight='bold')
axes[1].legend(ncol=2)
axes[1].grid(alpha=0.3)

plt.suptitle(f'Latent Space Visualization (t-SNE) - IMPROVED MASKING - Epoch {NUM_EPOCHS}', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_clustering.png'), dpi=300, bbox_inches='tight')
plt.show()

# Cluster purity analysis
print("\n" + "="*80)
print("CLUSTER PURITY ANALYSIS")
print("="*80)
print(f"\nCluster | Size | Dominant Type | Purity | Distribution")
print("-" * 80)
for i in range(8):
    mask = clusters == i
    cluster_types = labels[mask]
    if len(cluster_types) > 0:
        unique, counts = np.unique(cluster_types, return_counts=True)
        dominant = unique[np.argmax(counts)]
        purity = counts.max() / len(cluster_types)
        
        # Distribution
        dist = ", ".join([f"t{u+1}:{c}" for u, c in zip(unique, counts)])
        
        print(f"   {i}    | {len(cluster_types):4d} |      t{dominant+1}      | {purity:5.1%}  | {dist}")

print("="*80)

## Training Statistics

In [ ]:
print("="*80)
print("TRAINING STATISTICS")
print("="*80)
print(f"\nFinal Results:")
print(f"  Final Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Final Val Loss:   {history['val_loss'][-1]:.4f}")
print(f"  Best Val Loss:    {best_val_loss:.4f} (epoch {np.argmin(history['val_loss'])})")

print(f"\nImprovement:")
print(f"  Train Loss: {history['train_loss'][0]:.4f} → {history['train_loss'][-1]:.4f}")
print(f"  Val Loss:   {history['val_loss'][0]:.4f} → {history['val_loss'][-1]:.4f}")
print(f"  Reduction:  {(1 - history['val_loss'][-1]/history['val_loss'][0])*100:.1f}%")

print(f"\nTiming:")
print(f"  Total time:       {sum(history['epoch_time'])/60:.1f} minutes")
print(f"  Avg time/epoch:   {np.mean(history['epoch_time']):.1f} seconds")
print(f"  Est. for 100 epochs: {np.mean(history['epoch_time'])*100/3600:.1f} hours")

print(f"\nOutput Files:")
print(f"  Best model:        {os.path.join(OUTPUT_DIR, 'checkpoints', 'best_model.pt')}")
print(f"  Final model:       {os.path.join(OUTPUT_DIR, 'checkpoints', 'final_model.pt')}")
print(f"  Pretrained encoder: {os.path.join(OUTPUT_DIR, 'timesformer_encoder_pretrained.pt')}")
print("="*80)

## Next Steps

After pretraining completes:

1. **Compare clustering**: Check if t-SNE shows better separation than old masking
2. **Fine-tune classifier**: Use the pretrained encoder for classification
3. **Analyze attention**: Visualize what the model learned to focus on

The pretrained encoder is saved in:
```
output_improved/timesformer_encoder_pretrained.pt
```